In [ ]:
import os, sys, time, json, glob, math, random, warnings, subprocess, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import re
from scipy import stats
from scipy.stats import loguniform, randint, uniform

def _ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for _p in ("xgboost", "lightgbm", "catboost", "optuna"):
    _ensure(_p)

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeCV, Lasso, ElasticNet, LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import (mean_absolute_error, r2_score, make_scorer, roc_auc_score, average_precision_score)
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor, CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore", message=".*X does not have valid feature names.*", category=UserWarning)

In [ ]:
SEED = 42
N_TRIALS, SEEDS, K_GRID, N_ITER_RS = 40, (0, 1, 2, 3, 4), (0, 1, 2, 5, 10, 25, 50, 75, 100), 15
K_BROAD = (0, 10, 50)
K_ABLATION = (10, 25, 50)
HEAVY_Q = 0.80
OP_POINT_TOL = 1.10
RUN_CLUSTERING = True
N_FOLDS, MIN_TRAIN_FRAC, GAP, TRAIN_FRAC = 5, 0.5, 0, 0.8
ALERT_BUDGET = 0.10
N_BINS = 100
N_JOBS = os.cpu_count() or 1
np.random.seed(SEED); random.seed(SEED)

In [ ]:
COARSE_FILE = "/kaggle/input/datasets/poorvie05/polaris-2025-data/features_coarse.csv"
FINE_FILE   = "/kaggle/input/datasets/poorvie05/polaris-2025-data/features_fine.csv"
LOG_FILE    = "/kaggle/input/datasets/poorvie05/polaris-2025-data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv"
for f in (COARSE_FILE, FINE_FILE, LOG_FILE):
    assert os.path.exists(f), f"missing input file: {f}"

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
TABLES, FILES_WRITTEN = {}, {}

def show(df, n=None):
    d = df if n is None else df.head(n)
    print(d.round(3).to_string(index=False))
    
def save_table(name, df, cell):
    path = os.path.join(OUTPUT_DIR, name + ".csv")
    df.to_csv(path, index=False)
    TABLES[name] = df; FILES_WRITTEN[path] = cell
    return path
    
def save_fig(fig, name, cell):
    path = os.path.join(OUTPUT_DIR, name + ".png")
    fig.savefig(path, dpi=130, bbox_inches="tight")
    FILES_WRITTEN[path] = cell
    plt.show(); plt.close(fig)
    return path

In [ ]:
coarse = pd.read_csv(COARSE_FILE)
fine = pd.read_csv(FINE_FILE)
assert set(coarse.job_id) == set(fine.job_id) and coarse.job_id.is_unique and fine.job_id.is_unique

m = coarse[["job_id", "RUNTIME", "NODES", "NPROCS"]].merge(fine[["job_id", "RUNTIME", "NODES", "NPROCS"]], on="job_id", suffixes=("_c", "_f"))
for c in ("RUNTIME", "NODES", "NPROCS"):
    assert np.allclose(m[c + "_c"].fillna(-1), m[c + "_f"].fillna(-1)), c
df = (coarse.merge(fine.drop(columns=["RUNTIME", "NODES", "NPROCS"]), on="job_id", how="inner", validate="one_to_one")
            .sort_values("job_id").reset_index(drop=True))
print("merged shape:", df.shape, "| job_id sets identical: True | sorted ascending:", df.job_id.is_monotonic_increasing)

IO_COLS = ["POSIX_F_READ_TIME", "POSIX_F_WRITE_TIME", "POSIX_F_META_TIME",
           "MPIIO_F_READ_TIME", "MPIIO_F_WRITE_TIME", "MPIIO_F_META_TIME"]
READ_COLS = [f"read_{i}" for i in range(N_BINS)]
WRITE_COLS = [f"write_{i}" for i in range(N_BINS)]
_io = df[IO_COLS].sum(axis=1)
df = pd.concat([df, pd.DataFrame({"IO_TIME": _io, "y": np.log1p(_io)})], axis=1)

CTX_NUM = ["WALLTIME_SECONDS", "NODES_REQUESTED", "CORES_REQUESTED", "GPUS_REQUESTED"]
CTX_CAT = ["USERNAME_GENID", "PROJECT_NAME_GENID", "QUEUE_NAME", "ALLOCATION_AWARD_CATEGORY", "SCIENCE_FIELD_SHORT"]
log = pd.read_csv(LOG_FILE, usecols=["JOB_NAME", "RUNTIME_SECONDS", "QUEUED_TIMESTAMP", "END_TIMESTAMP"] + CTX_NUM + CTX_CAT, low_memory=False)
log["job_id"] = pd.to_numeric(log["JOB_NAME"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")   # the job id is in JOB_NAME (COBALT_JOBID is 0)
n_log, n_dup = len(log), int(log["job_id"].duplicated().sum())
log = log.dropna(subset=["job_id"]).drop_duplicates("job_id", keep="first").copy()                      # repeated ids: keep the first record
log["job_id"] = log["job_id"].astype("int64")
_epoch = pd.Timestamp("1970-01-01")
# QUEUED/END times are used ONLY to order other jobs' labels in time (a job's history = jobs that ended before it was queued); never as features.
log["_QUEUED_TS"] = (pd.to_datetime(log["QUEUED_TIMESTAMP"]) - _epoch) / pd.Timedelta(seconds=1)
log["_END_TS"] = (pd.to_datetime(log["END_TIMESTAMP"]) - _epoch) / pd.Timedelta(seconds=1)
df = df.merge(log[["job_id", "RUNTIME_SECONDS", "_QUEUED_TS", "_END_TS"] + CTX_NUM + CTX_CAT], on="job_id", how="left", validate="one_to_one")
hit = df["RUNTIME_SECONDS"].notna()
assert (df.loc[hit, "RUNTIME"] - df.loc[hit, "RUNTIME_SECONDS"]).abs().max() < 1, "log RUNTIME disagrees with the features file: wrong join"
df = df.drop(columns="RUNTIME_SECONDS")       # post-run quantity: used only for the join check above
print(f"scheduler log: {n_log} records, {n_dup} repeated ids dropped | our jobs found in the log: {int(hit.sum())} of {len(df)}")

N_TOTAL = len(df)
N_TRAIN = int(N_TOTAL * TRAIN_FRAC)

print(f"jobs with IO_TIME<=0 (zero-I/O jobs): {(df.IO_TIME <= 0).sum()}  [claim: none]")
print(f"median IO_TIME: {df.IO_TIME.median():.2f} s  [claim: ~44]")
print(f"max IO_TIME: {df.IO_TIME.max():.3e} s  [claim: ~7.8e6]")
print(f"mean y train={df.y.iloc[:N_TRAIN].mean():.3f}  test={df.y.iloc[N_TRAIN:].mean():.3f}  [claim: ~4.35 -> ~3.72]")

print("\n missing values (non-zero only) ")
na = df.isna().sum(); print(na[na > 0].to_string() if (na > 0).any() else "none")
print("\n INTERFACE counts "); print(df.INTERFACE.value_counts().to_string())
print("\n BIN_WIDTH distribution "); print(df.BIN_WIDTH.value_counts().sort_index().to_string())

allbins = df[READ_COLS + WRITE_COLS].to_numpy(float)
assert np.isfinite(allbins).all() and (allbins >= 0).all(), "bins must be finite and non-negative"
ratio = allbins.sum(axis=1) / df.TOTAL_IO_BYTES.to_numpy(float)
print(f"\nsum(read bins)+sum(write bins) / TOTAL_IO_BYTES: min={ratio.min():.6f} median={np.median(ratio):.6f} max={ratio.max():.6f}")
assert np.abs(ratio - 1).max() < 1e-3

cov = 100 * df.BIN_WIDTH / df.RUNTIME
print("fraction of runtime covered by all 100 bins: quantiles", np.round(cov.quantile([0, .05, .5, .95, 1]).to_numpy(), 3))

# Exploratory Data Analysis

In [ ]:
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
COL = {"POSIX": "#1f77b4", "MPIIO": "#d95f02"}
tot_bytes = allbins.sum(axis=1)
logb = np.log10(tot_bytes + 1)

In [ ]:
# distribution of y
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df.y, bins=60, color="#555"); ax[0].set(title="log1p(IO_TIME), all jobs", xlabel="log1p(IO_TIME [s])", ylabel="jobs")
for itf, g in df.groupby("INTERFACE", observed=True):
    ax[1].hist(g.y, bins=60, alpha=.55, color=COL[itf], label=f"{itf} (n={len(g)})")
ax[1].set(title="by INTERFACE", xlabel="log1p(IO_TIME [s])"); ax[1].legend()
save_fig(fig, "target_distribution", 2)
print(df.groupby("INTERFACE", observed=True).y.describe().round(3).to_string())

In [ ]:
# log total bytes vs y
rho = float(stats.spearmanr(logb, df.y).statistic)
fig, ax = plt.subplots(figsize=(6.5, 5))
for itf, g in df.groupby("INTERFACE", observed=True):
    ax.scatter(logb[g.index], g.y, s=6, alpha=.35, color=COL[itf], label=itf)
ax.set(xlabel="log10(total bytes read+written + 1)", ylabel="log1p(IO_TIME [s])", title=f"Total bytes vs I/O time (Spearman={rho:.3f})"); ax.legend(markerscale=3)
save_fig(fig, "bytes_vs_target", 2)
print(f"Spearman(log bytes, y): all={rho:.3f}", {i: round(float(stats.spearmanr(logb[g.index], g.y).statistic), 3) for i, g in df.groupby("INTERFACE", observed=True)})

In [ ]:
# example timelines: 3 sizes (10th/50th/90th pct of bytes) x 2 interfaces
picks = []
for itf in ("POSIX", "MPIIO"):
    g = df[(df.INTERFACE == itf) & (logb > 0)]
    lg = pd.Series(logb, index=df.index)[g.index]
    for q in (.10, .50, .90):
        picks.append(int((lg - lg.quantile(q)).abs().idxmin()))
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for ax, ix in zip(axes.ravel(), picks):
    r = df.loc[ix, READ_COLS].to_numpy(float); w = df.loc[ix, WRITE_COLS].to_numpy(float)
    ax.step(range(N_BINS), np.where(r > 0, r, np.nan), where="mid", color="#1f77b4", label="read")
    ax.step(range(N_BINS), np.where(w > 0, w, np.nan), where="mid", color="#d95f02", label="write")
    ax.set_yscale("log"); ax.set_title(f"job {df.loc[ix,'job_id']} {df.loc[ix,'INTERFACE']}, {tot_bytes[ix]/2**30:.2f} GiB, bin {df.loc[ix,'BIN_WIDTH']}s", fontsize=9)
for ax in axes[1]: ax.set_xlabel("time bin")
for ax in axes[:, 0]: ax.set_ylabel("bytes per bin")
axes[0, 0].legend()
save_fig(fig, "example_timelines", 2)

In [ ]:
# drift: mean y over 20 equal job_id blocks
blocks = np.array_split(np.arange(N_TOTAL), 20)
bm = pd.DataFrame({"block": range(1, 21), "job_id_mean": [df.job_id.iloc[b].mean() for b in blocks], "mean_y": [df.y.iloc[b].mean() for b in blocks]})
boundary_id = df.job_id.iloc[N_TRAIN - 1] / 2 + df.job_id.iloc[N_TRAIN] / 2
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bm.job_id_mean, bm.mean_y, "o-", color="#333"); ax.axvline(boundary_id, color="crimson", ls="--", label="train | test boundary")
ax.set(xlabel="job_id (block mean)", ylabel="mean log1p(IO_TIME)", title="Drift of the target over time (20 equal blocks)"); ax.legend()
save_fig(fig, "target_drift", 2)
show(bm)

In [ ]:
# BIN_WIDTH distribution
ct = pd.crosstab(df.BIN_WIDTH, df.INTERFACE)
fig, ax = plt.subplots(figsize=(8, 4)); bottom = np.zeros(len(ct)); xs = np.arange(len(ct))
for itf in ct.columns:
    ax.bar(xs, ct[itf].to_numpy(), bottom=bottom, color=COL[itf], label=itf); bottom += ct[itf].to_numpy()
ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in ct.index], rotation=45); ax.set(xlabel="BIN_WIDTH [s]", ylabel="jobs", title="BIN_WIDTH distribution"); ax.legend()
save_fig(fig, "bin_width_distribution", 2)

In [ ]:
COARSE_TOTALS = ["POSIX_BYTES_READ", "POSIX_BYTES_WRITTEN", "MPIIO_BYTES_READ", "MPIIO_BYTES_WRITTEN",
                 "POSIX_READS", "POSIX_WRITES", "MPIIO_INDEP_READS", "MPIIO_INDEP_WRITES",
                 "MPIIO_COLL_READS", "MPIIO_COLL_WRITES", "TOTAL_IO_BYTES"]
FORBIDDEN = set(IO_COLS + COARSE_TOTALS + ["EFFECTIVE_IO_BW_RUNTIME", "EFFECTIVE_IO_BW_IOTIME",
                "RUNTIME", "BIN_WIDTH", "TIME_TO_FIRST_IO", "CORES_USED",
                "IO_TIME", "y", "job_id"])  # job_id excluded too: no history/time-position features are used
BASE_ALLOWED = ["NODES", "NPROCS", "INTERFACE"]
SUBMIT_ALLOWED = CTX_NUM + CTX_CAT
LOG_FORBIDDEN = ["RUNTIME_SECONDS", "EXIT_STATUS", "EXIT_CODE", "START_TIMESTAMP", "END_TIMESTAMP", "QUEUED_WAIT_SECONDS", "ELIGIBLE_WAIT_SECONDS",
                 "USED_CORE_HOURS", "OVERBURN_CORE_HOURS", "NODES_USED", "TASK_EXIT_CODES", "_END_TS", "_QUEUED_TS"]
FORBIDDEN |= set(LOG_FORBIDDEN)
assert not (set(SUBMIT_ALLOWED) & FORBIDDEN)

def allowed_source_columns(k):
    return BASE_ALLOWED + [f"read_{i}" for i in range(k)] + [f"write_{i}" for i in range(k)]

def assert_allowed_source(cols, k):
    cols = set(cols)
    bad = cols & FORBIDDEN
    assert not bad, f"FORBIDDEN columns used: {sorted(bad)}"          # whole-run quantities would leak the target
    extra = cols - set(allowed_source_columns(k))
    assert not extra, f"columns outside the allowed set for k={k}: {sorted(extra)[:5]}"  # includes future bins >= k

_NAME_OK = re.compile(r"^(log_nodes|log_nprocs|is_mpiio|log_read_sofar|log_write_sofar|log_total_sofar|read_share|n_active|"
                      r"first_active|last_active|peak_share|burstiness|seg\d_[rw]|rawr_\d+|raww_\d+|clu_\d+|clu_dist|"
                      r"ctx_log_walltime|ctx_log_nodes_req|ctx_log_cores_req|ctx_gpus_req|ctx_enc_user|ctx_enc_project|ctx_enc_queue|"
                      r"ctx_enc_category|ctx_enc_field|ctx_n_user|ctx_n_project)$")
def assert_features_clean(names):
    bad = [n for n in names if not _NAME_OK.match(n)]
    assert not bad, f"non-whitelisted feature names reach the model: {bad[:5]}"

# self-check that the assert actually fires
for _cols, _k in ((["RUNTIME"], 10), (["read_10"], 10), (["BIN_WIDTH", "NODES"], 0), (["TOTAL_IO_BYTES"], 100)):
    try:
        assert_allowed_source(_cols, _k); raise RuntimeError("assert did not fire")
    except AssertionError:
        pass
assert_allowed_source(allowed_source_columns(100), 100)

In [ ]:
def positional_folds(n, n_folds=5, min_train_frac=0.5, gap=0):
    edges = np.linspace(int(n * min_train_frac), n, n_folds + 1).astype(int)
    return [(np.arange(0, edges[i] - gap), np.arange(edges[i], edges[i + 1])) for i in range(n_folds)]

assert N_TRAIN == int(len(df) * TRAIN_FRAC)
TRAIN_DF = df.iloc[:N_TRAIN].copy()      # first 80% by job_id: the ONLY data used for any choice
TEST_DF = df.iloc[N_TRAIN:].copy()       # last 20%: scored once per final model, never used to select anything
y_train, y_test = TRAIN_DF.y.to_numpy(), TEST_DF.y.to_numpy()
FOLDS = positional_folds(N_TRAIN, N_FOLDS, MIN_TRAIN_FRAC, GAP)

FIT_STATS = {"nodes_median": float(TRAIN_DF.NODES.median())}
print(f"train={len(TRAIN_DF)} (job_id {TRAIN_DF.job_id.min()}..{TRAIN_DF.job_id.max()})  test={len(TEST_DF)} (job_id {TEST_DF.job_id.min()}..{TEST_DF.job_id.max()})")

# ---- submission-time context table (built once; uses TRAIN labels of strictly earlier, already-finished jobs only) ----
CTX_FEATURE_NAMES = ["ctx_log_walltime", "ctx_log_nodes_req", "ctx_log_cores_req", "ctx_gpus_req",
                     "ctx_enc_user", "ctx_enc_project", "ctx_enc_queue", "ctx_enc_category", "ctx_enc_field", "ctx_n_user", "ctx_n_project"]

def build_context_table(df, y_tr, n_train, smooth=10.0):
    assert not (set(SUBMIT_ALLOWED) & FORBIDDEN)
    n = len(df); out = pd.DataFrame(index=df.index)
    med = df.iloc[:n_train][CTX_NUM].median()
    x = df[CTX_NUM].fillna(med)
    out["ctx_log_walltime"] = np.log1p(x["WALLTIME_SECONDS"].clip(lower=0))
    out["ctx_log_nodes_req"] = np.log1p(x["NODES_REQUESTED"].clip(lower=0))
    out["ctx_log_cores_req"] = np.log1p(x["CORES_REQUESTED"].clip(lower=0))
    out["ctx_gpus_req"] = x["GPUS_REQUESTED"]
    q = df["_QUEUED_TS"].to_numpy(float); e_tr = df["_END_TS"].to_numpy(float)[:n_train]
    q = np.where(np.isnan(q), -np.inf, q)
    e_tr = np.where(np.isnan(e_tr), np.inf, e_tr)
    yp_all = np.asarray(y_tr, float)
    codes = {}
    for c in CTX_CAT:
        s = df[c].astype("object").where(df[c].notna(), "unknown").astype(str)
        seen = {v: j for j, v in enumerate(s.iloc[:n_train].unique())}
        codes[c] = s.map(seen).fillna(-1).astype(int).to_numpy()
    enc = {c: np.zeros(n) for c in CTX_CAT}
    cnt = {c: np.zeros(n) for c in ("USERNAME_GENID", "PROJECT_NAME_GENID")}
    pos_tr = np.arange(n_train)
    for i in range(n):
        pool = np.flatnonzero((pos_tr < i) & (e_tr < q[i]))
        if pool.size == 0: continue
        yp = yp_all[pool]; gm = yp.mean()
        for c in CTX_CAT:
            ci = codes[c][i]
            same = (codes[c][pool] == ci) if ci >= 0 else np.zeros(pool.size, bool)
            nn = int(same.sum())
            enc[c][i] = (yp[same].sum() - nn * gm) / (nn + smooth)
            if c in cnt: cnt[c][i] = nn
    out["ctx_enc_user"], out["ctx_enc_project"] = enc["USERNAME_GENID"], enc["PROJECT_NAME_GENID"]
    out["ctx_enc_queue"], out["ctx_enc_category"], out["ctx_enc_field"] = enc["QUEUE_NAME"], enc["ALLOCATION_AWARD_CATEGORY"], enc["SCIENCE_FIELD_SHORT"]
    out["ctx_n_user"], out["ctx_n_project"] = np.log1p(cnt["USERNAME_GENID"]), np.log1p(cnt["PROJECT_NAME_GENID"])
    return out[CTX_FEATURE_NAMES]

CTX = build_context_table(df, y_train, N_TRAIN)

In [ ]:
FEATURE_SETS = ("totals", "shape", "segments", "raw")

def build_early_features(df, k, feature_set="shape", fit_stats=None):
    use_ctx = feature_set.endswith("+ctx")                         # "+ctx" adds the submission-time context block
    feature_set = feature_set[:-4] if use_ctx else feature_set
    if feature_set not in FEATURE_SETS: raise ValueError(feature_set)
    if not 0 <= k <= N_BINS: raise ValueError(k)
    cols = allowed_source_columns(k)
    assert_allowed_source(cols, k)                 # leakage gate: only NODES, NPROCS, INTERFACE, bins < k are ever read
    d = df[cols]
    nodes = d["NODES"].to_numpy(float)
    if fit_stats is not None:
        nodes = np.where(np.isnan(nodes), fit_stats["nodes_median"], nodes)   # train-only median
    f = {"log_nodes": np.log1p(nodes), "log_nprocs": np.log1p(d["NPROCS"].to_numpy(float)),
         "is_mpiio": (d["INTERFACE"].to_numpy() == "MPIIO").astype(float)}
    if k > 0:
        R = d[[f"read_{i}" for i in range(k)]].to_numpy(float)
        W = d[[f"write_{i}" for i in range(k)]].to_numpy(float)
        T = R + W
        r_sum, w_sum, t_sum = R.sum(1), W.sum(1), T.sum(1)
        if feature_set == "raw":                    # ablation only
            for i in range(k): f[f"rawr_{i}"] = np.log1p(R[:, i])
            for i in range(k): f[f"raww_{i}"] = np.log1p(W[:, i])
        else:
            f["log_read_sofar"], f["log_write_sofar"], f["log_total_sofar"] = np.log1p(r_sum), np.log1p(w_sum), np.log1p(t_sum)
            if feature_set in ("shape", "segments"):
                pos = t_sum > 0
                active = T > 0
                n_act = active.sum(1)
                has = n_act > 0
                peak = T.max(1)
                f["read_share"] = np.divide(r_sum, t_sum, out=np.zeros(len(df)), where=pos)
                f["n_active"] = n_act.astype(float)
                f["first_active"] = np.where(has, active.argmax(1), k) / k            # k/k=1 if nothing seen yet
                f["last_active"] = np.where(has, k - 1 - active[:, ::-1].argmax(1), 0) / k
                f["peak_share"] = np.divide(peak, t_sum, out=np.zeros(len(df)), where=pos)
                f["burstiness"] = np.divide(peak * n_act, t_sum, out=np.zeros(len(df)), where=pos)  # peak / mean over active bins
            if feature_set == "segments" and k >= 5:
                for s, ix in enumerate(np.array_split(np.arange(k), 5)):
                    f[f"seg{s}_r"], f[f"seg{s}_w"] = np.log1p(R[:, ix].sum(1)), np.log1p(W[:, ix].sum(1))
    out = pd.DataFrame(f, index=df.index)
    if use_ctx:
        out = pd.concat([out, CTX.loc[df.index, CTX_FEATURE_NAMES]], axis=1)   # precomputed from TRAIN labels of earlier, finished jobs only
    assert_features_clean(out.columns)
    return out

In [ ]:
def run_feature_unit_tests(n_rows=300):
    rng = np.random.default_rng(SEED)
    base = TRAIN_DF.iloc[:n_rows].copy()
    n_ok = 0
    for k in (0, 1, 5, 10, 50, 100):
        for fs in FEATURE_SETS + tuple(s_ + "+ctx" for s_ in FEATURE_SETS):
            ref = build_early_features(base, k, fs, FIT_STATS)
            # (1) no bin index >= k: overwrite later bins with junk -> features must not change
            alt = base.copy()
            for i in range(k, N_BINS):
                alt[f"read_{i}"] = rng.random(n_rows) * 1e12; alt[f"write_{i}"] = rng.random(n_rows) * 1e12
            # (2) no forbidden column: scramble every forbidden column too -> features must not change
            for c in FORBIDDEN & set(alt.columns):
                if c not in ("INTERFACE",) and pd.api.types.is_numeric_dtype(alt[c]):
                    alt[c] = rng.random(n_rows) * 1e9
            np.testing.assert_allclose(ref.to_numpy(), build_early_features(alt, k, fs, FIT_STATS).to_numpy(), equal_nan=True)
            # (3) shape, finiteness, expected column count
            assert len(ref) == n_rows and np.isfinite(ref.to_numpy()).all()
            expect = 3 if k == 0 else {"totals": 6, "shape": 12, "segments": 12 + (10 if k >= 5 else 0), "raw": 3 + 2 * k}[fs.replace("+ctx", "")]
            expect += len(CTX_FEATURE_NAMES) if fs.endswith("+ctx") else 0
            assert ref.shape[1] == expect, (k, fs, ref.shape[1], expect)
            n_ok += 1
    # the leakage gate itself must reject a future bin and a forbidden column
    for bad in (["NODES", "read_10"], ["RUNTIME"]):
        try: assert_allowed_source(bad, 10); raise RuntimeError("gate did not fire")
        except AssertionError: pass
    return n_ok
ex = build_early_features(TRAIN_DF.iloc[:3], 25, "shape", FIT_STATS); show(ex.reset_index(drop=True))

In [ ]:
def run_context_tests():
    rng = np.random.default_rng(SEED)
    base = build_context_table(df, y_train, N_TRAIN)
    np.testing.assert_allclose(base.to_numpy(float), CTX.to_numpy(float), equal_nan=True)
    p = 2500
    y2 = y_train.copy(); y2[p:] = rng.normal(8, 3, N_TRAIN - p)                  # change the labels of every train job at position >= p
    alt = build_context_table(df, y2, N_TRAIN)
    # (1) causality: features of jobs at position <= p cannot depend on labels at position >= p
    np.testing.assert_allclose(alt.iloc[:p + 1].to_numpy(float), base.iloc[:p + 1].to_numpy(float), equal_nan=True)
    # (2) the check is not vacuous: later jobs DO depend on past labels
    assert not np.allclose(alt.iloc[p + 50:N_TRAIN].to_numpy(float), base.iloc[p + 50:N_TRAIN].to_numpy(float), equal_nan=True)
    # (3) test labels cannot enter: build_context_table never receives them (only y_train is passed)
    assert not (set(SUBMIT_ALLOWED) & FORBIDDEN)
    return True

In [ ]:
def evaluate(y_true, y_pred, name=""):
    yt = np.asarray(y_true, float); yp = np.clip(np.asarray(y_pred, float), 0, 30)
    ae = np.abs(yp - yt)
    mdape = np.median(np.abs(np.expm1(yp) - np.expm1(yt)) / np.maximum(np.expm1(yt), 1.0))
    return {"name": name, "MAE": ae.mean(), "MedAE": np.median(ae), "R2": r2_score(yt, yp),
            "MdAPE": mdape, "within_2x": float(np.mean(ae <= np.log(2)))}

In [ ]:
def clipped_mae(y_true, y_pred): return mean_absolute_error(y_true, np.clip(y_pred, 0, 30))
MAE_SCORER = make_scorer(clipped_mae, greater_is_better=False)

In [ ]:
_XC = {}
def get_data(k, fs):
    key = (k, fs)
    if key not in _XC:
        Xcv = build_early_features(TRAIN_DF, k, fs, None)
        Xtr = build_early_features(TRAIN_DF, k, fs, FIT_STATS)
        Xte = build_early_features(TEST_DF, k, fs, FIT_STATS)
        assert_features_clean(Xcv.columns)
        _XC[key] = dict(cols=list(Xcv.columns), cv=Xcv.to_numpy(float), tr=Xtr.to_numpy(float), te=Xte.to_numpy(float), tr_df=Xtr, te_df=Xte)
    return _XC[key]

In [ ]:
def cv_mae(est, X, y, folds=None):
    folds = FOLDS if folds is None else folds
    return float(np.mean([clipped_mae(y[va], clone(est).fit(X[tr], y[tr]).predict(X[va])) for tr, va in folds]))

In [ ]:
def pipe(model, scale=False):
    # imputer sits first so any NaN (only NODES) is filled with the median of the TRAIN part of each fold
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler() if scale else "passthrough"), ("model", model)])

In [ ]:
def baseline_global_median():
    cv = np.mean([clipped_mae(y_train[va], np.full(len(va), np.median(y_train[tr]))) for tr, va in FOLDS])
    ev = evaluate(y_test, np.full(len(y_test), np.median(y_train)), "global_median")   # train median only
    return {"cv_mae": float(cv), **ev}
def baseline_global_mean():
    # mean of y on the log1p scale; fitted on the training part only (inside each fold for CV)
    cv = np.mean([clipped_mae(y_train[va], np.full(len(va), np.mean(y_train[tr]))) for tr, va in FOLDS])
    ev = evaluate(y_test, np.full(len(y_test), np.mean(y_train)), "global_mean")
    return {"cv_mae": float(cv), **ev}

def baseline_group_median(col):
    lab = lambda part: part[col].astype("object").where(part[col].notna(), "unknown").astype(str).to_numpy()
    g_tr, g_te = lab(TRAIN_DF), lab(TEST_DF)
    def pred(y_part, g_part, g_new):
        med = pd.Series(y_part).groupby(g_part).median()
        return pd.Series(g_new).map(med).fillna(np.median(y_part)).to_numpy()
    cv = np.mean([clipped_mae(y_train[va], pred(y_train[tr], g_tr[tr], g_tr[va])) for tr, va in FOLDS])
    ev = evaluate(y_test, pred(y_train, g_tr, g_te), col + "_median")
    return {"cv_mae": float(cv), **ev}

BASELINES = {"global_median": baseline_global_median(), "global_mean": baseline_global_mean(),
             "user_median": baseline_group_median("USERNAME_GENID"), "project_median": baseline_group_median("PROJECT_NAME_GENID")}
bl = pd.DataFrame([{**{"baseline": n}, **{c: v for c, v in r.items() if c != "name"}} for n, r in BASELINES.items()])
show(bl)
save_table("baselines", bl, 6)

In [ ]:
def hgb_fixed(seed): return pipe(HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, max_depth=4, random_state=seed))

rows = []
for k in K_ABLATION:
    for fs in FEATURE_SETS + tuple(s_ + "+ctx" for s_ in FEATURE_SETS):
        d = get_data(k, fs)
        cv = cv_mae(hgb_fixed(SEED), d["cv"], y_train)
        te = [clipped_mae(y_test, hgb_fixed(s).fit(d["tr"], y_train).predict(d["te"])) for s in SEEDS]   # test is only REPORTED here
        rows.append({"k": k, "feature_set": fs, "n_features": d["tr"].shape[1], "cv_mae": cv, "test_mae": float(np.mean(te))})
abl = pd.DataFrame(rows)
show(abl)
summ = abl.groupby("feature_set", observed=True).agg(cv_mae_mean=("cv_mae", "mean"), test_mae_mean=("test_mae", "mean")).reset_index().sort_values("cv_mae_mean")
print("\nmean over k in", K_ABLATION); show(summ)

FEATURE_SETS_COMPARE = ("totals+ctx", "shape+ctx", "segments+ctx")
FEATURE_SET = None
print(f"\nfeature sets carried forward: {FEATURE_SETS_COMPARE} (the primary one is selected by CV only)")
save_table("feature_set_ablation", abl, 7)

# Broad Model Comparison

In [ ]:
MODELS = ["Ridge", "Lasso", "ElasticNet", "KNN", "RandomForest", "ExtraTrees", "HistGB", "XGBoost", "LightGBM", "CatBoost"]
BOOSTING = ("HistGB", "XGBoost", "LightGBM", "CatBoost")
DETERMINISTIC = ("Ridge", "Lasso", "ElasticNet", "KNN")
SCALED = ("Ridge", "Lasso", "ElasticNet", "KNN")

class ResidualRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, estimator=None, alphas=(0.01, 0.1, 1, 10, 100, 1000)):
        self.estimator = estimator; self.alphas = alphas
    def fit(self, X, y):
        y = np.asarray(y, float)
        self.baseline_ = pipe(RidgeCV(alphas=self.alphas), scale=True).fit(X, y)
        self.estimator_ = clone(self.estimator).fit(X, y - self.baseline_.predict(X))
        return self
    def predict(self, X):
        return self.baseline_.predict(X) + self.estimator_.predict(X)

In [ ]:
def base_model(name, p, seed):
    p = dict(p)
    if name == "Ridge": return Ridge(**p)
    if name == "Lasso": return Lasso(max_iter=20000, **p)
    if name == "ElasticNet": return ElasticNet(max_iter=20000, **p)
    if name == "KNN": return KNeighborsRegressor(n_jobs=N_JOBS, **p)
    if name == "RandomForest": return RandomForestRegressor(n_jobs=N_JOBS, random_state=seed, **p)
    if name == "ExtraTrees": return ExtraTreesRegressor(n_jobs=N_JOBS, random_state=seed, **p)
    if name == "HistGB": return HistGradientBoostingRegressor(random_state=seed, **p)
    if name == "XGBoost": return XGBRegressor(n_jobs=N_JOBS, random_state=seed, tree_method="hist", verbosity=0, **p)
    if name == "LightGBM": return LGBMRegressor(n_jobs=N_JOBS, random_state=seed, verbose=-1, subsample_freq=1, **p)
    if name == "CatBoost": return CatBoostRegressor(thread_count=N_JOBS, random_seed=seed, verbose=0, allow_writing_files=False, **p)
    raise ValueError(name)

In [ ]:
def make_estimator(name, params, seed, mode="direct"):
    est = pipe(base_model(name, params, seed), scale=name in SCALED)
    return est if mode == "direct" else ResidualRegressor(est)

In [ ]:
RS_SPACES = {
    "Ridge": {"alpha": loguniform(1e-3, 1e3)},
    "Lasso": {"alpha": loguniform(1e-4, 1)},
    "ElasticNet": {"alpha": loguniform(1e-4, 1), "l1_ratio": uniform(0.1, 0.85)},
    "KNN": {"n_neighbors": randint(3, 80), "weights": ["uniform", "distance"], "p": [1, 2]},
    "RandomForest": {"n_estimators": randint(100, 401), "max_depth": [4, 6, 8, 12, None], "min_samples_leaf": randint(1, 25), "max_features": uniform(0.3, 0.7)},
    "ExtraTrees": {"n_estimators": randint(100, 401), "max_depth": [4, 6, 8, 12, None], "min_samples_leaf": randint(1, 25), "max_features": uniform(0.3, 0.7)},
}
def suggest_params(name, t):
    if name == "HistGB":
        return dict(loss=t.suggest_categorical("loss", ["absolute_error", "squared_error"]), max_iter=t.suggest_int("max_iter", 100, 500),
                    learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True), max_depth=t.suggest_int("max_depth", 2, 8),
                    min_samples_leaf=t.suggest_int("min_samples_leaf", 5, 60), l2_regularization=t.suggest_float("l2_regularization", 1e-3, 10, log=True))
    if name == "XGBoost":
        return dict(objective=t.suggest_categorical("objective", ["reg:absoluteerror", "reg:squarederror"]), n_estimators=t.suggest_int("n_estimators", 100, 500),
                    learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True), max_depth=t.suggest_int("max_depth", 2, 8),
                    subsample=t.suggest_float("subsample", 0.5, 1.0), colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
                    min_child_weight=t.suggest_int("min_child_weight", 1, 20), reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True))
    if name == "LightGBM":
        return dict(objective=t.suggest_categorical("objective", ["l1", "l2"]), n_estimators=t.suggest_int("n_estimators", 100, 500),
                    learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True), num_leaves=t.suggest_int("num_leaves", 4, 64),
                    min_child_samples=t.suggest_int("min_child_samples", 5, 60), subsample=t.suggest_float("subsample", 0.5, 1.0),
                    colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0), reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True))
    if name == "CatBoost":
        return dict(loss_function=t.suggest_categorical("loss_function", ["MAE", "RMSE"]), iterations=t.suggest_int("iterations", 100, 400),
                    learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True), depth=t.suggest_int("depth", 3, 8),
                    l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1, 10, log=True))
    raise ValueError(name)

In [ ]:
TUNE_CACHE = {}
def tune_config(name, mode, k, fs):
    key = (name, mode, k, fs, N_TRIALS, N_ITER_RS)
    if key in TUNE_CACHE: return TUNE_CACHE[key]
    X = get_data(k, fs)["cv"]
    if name in BOOSTING:
        sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=max(3, N_TRIALS // 3))
        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(lambda t: cv_mae(make_estimator(name, suggest_params(name, t), SEED, mode), X, y_train), n_trials=N_TRIALS)
        res = {"params": dict(study.best_params), "cv_mae": float(study.best_value)}
    else:
        prefix = "model__" if mode == "direct" else "estimator__model__"
        dist = {prefix + a: b for a, b in RS_SPACES[name].items()}
        rs = RandomizedSearchCV(make_estimator(name, {}, SEED, mode), dist, n_iter=N_ITER_RS, cv=FOLDS, scoring=MAE_SCORER,
                                random_state=SEED, n_jobs=1, refit=False).fit(X, y_train)
        res = {"params": {a[len(prefix):]: (v.item() if hasattr(v, "item") else v) for a, v in rs.best_params_.items()}, "cv_mae": float(-rs.best_score_)}
    TUNE_CACHE[key] = res
    return res

In [ ]:
def fit_final(name, mode, params, k, fs, seeds=None):
    d = get_data(k, fs); seeds = SEEDS if seeds is None else seeds
    use = seeds[:1] if name in DETERMINISTIC else seeds
    models = [make_estimator(name, params, s, mode).fit(d["tr"], y_train) for s in use]
    return models, np.mean([m.predict(d["te"]) for m in models], axis=0)

In [ ]:
def canon_fs(k, fs):
    ctx = fs.endswith("+ctx"); b = fs[:-4] if ctx else fs
    if k == 0: b = "totals"
    elif b == "segments" and k < 5: b = "shape"
    return b + "+ctx" if ctx else b

In [ ]:
def run_comparison(models, modes, ks, fs, tag=""):
    rows = []
    for k in ks:
        t0 = time.perf_counter()
        for name in models:
            for mode in modes:
                fe = canon_fs(k, fs)
                tc = tune_config(name, mode, k, fe)
                _, pred = fit_final(name, mode, tc["params"], k, fe)      # test scored once per final model, for reporting only
                ev = evaluate(y_test, pred, name)
                rows.append({"k": k, "model": name, "mode": mode, "cv_mae": tc["cv_mae"], "test_mae": ev["MAE"], "MedAE": ev["MedAE"],
                             "R2": ev["R2"], "MdAPE": ev["MdAPE"], "within_2x": ev["within_2x"], "params": json.dumps(tc["params"], default=str)})
        print(f"  k={k} done in {time.perf_counter()-t0:.0f}s", flush=True)
    out = pd.DataFrame(rows)
    # winner per k is decided by CV MAE alone
    out["cv_winner"] = out.groupby("k")["cv_mae"].transform("min") == out["cv_mae"]
    return out

BROAD_SETS = FEATURE_SETS_COMPARE
parts = []
for fs in BROAD_SETS:
    print(f"feature_set = {fs}")
    part = run_comparison(MODELS, ("direct", "residual"), K_BROAD, fs)
    part.insert(0, "feature_set", fs); parts.append(part)
BROAD = pd.concat(parts, ignore_index=True)

BROAD["cv_winner_overall"] = BROAD.groupby("k")["cv_mae"].transform("min") == BROAD["cv_mae"]
for k in K_BROAD:
    t = BROAD[BROAD.k == k]
    pick = t.loc[t.groupby(["feature_set", "model"])["cv_mae"].idxmin()]       # each model's CV-best mode
    cvp = pick.pivot(index="model", columns="feature_set", values="cv_mae").reindex(MODELS)[list(BROAD_SETS)]
    tep = pick.pivot(index="model", columns="feature_set", values="test_mae").reindex(MODELS)[list(BROAD_SETS)]
    print(f"\n=== k={k}: each model's CV-best mode, by feature set ===")
    print(pd.concat({"CV MAE": cvp, "test MAE": tep}, axis=1).round(3).to_string())
    w = t[t["cv_winner_overall"]].iloc[0]
    print(f"overall CV winner at k={k}: {w['model']}/{w['mode']} with '{w['feature_set']}' (CV {w['cv_mae']:.3f}, test {w['test_mae']:.3f})")
save_table("broad_model_comparison", BROAD, 8)

# Value Of Information Curve

In [ ]:
SHORTLIST = ["Ridge", "HistGB", "CatBoost", "LightGBM"]

def choose_op_k(cv_by_k, tol=OP_POINT_TOL):
    ks = sorted(cv_by_k); ref = cv_by_k[ks[-1]]
    return min(k for k in ks if cv_by_k[k] <= tol * ref)

FINAL_BY_FS, rows = {}, []
for fs in FEATURE_SETS_COMPARE:
    FINAL_BY_FS[fs] = {}
    for k in K_GRID:
        t0 = time.perf_counter(); fe = canon_fs(k, fs)
        cands = [(n, m, tune_config(n, m, k, fe)) for n in SHORTLIST for m in ("direct", "residual")]
        name, mode, tc = min(cands, key=lambda c: c[2]["cv_mae"])
        models, pred = fit_final(name, mode, tc["params"], k, fe)
        ev = evaluate(y_test, pred, name)
        FINAL_BY_FS[fs][k] = dict(model=name, mode=mode, params=tc["params"], cv_mae=tc["cv_mae"], models=models, test_pred=pred, metrics=ev)
        rows.append({"feature_set": fs, "k": k, "model": name, "mode": mode, "cv_mae": tc["cv_mae"], "test_mae": ev["MAE"], "MedAE": ev["MedAE"],
                     "R2": ev["R2"], "MdAPE": ev["MdAPE"], "within_2x": ev["within_2x"], "params": json.dumps(tc["params"], default=str)})
        print(f"  [{fs}] k={k}: winner {name}/{mode}  CV {tc['cv_mae']:.3f}  test {ev['MAE']:.3f}  ({time.perf_counter()-t0:.0f}s)", flush=True)

def add_gap(g):
    g = g.sort_values("k").copy()
    g["gap_closed_cv_%"] = 100 * (g.cv_mae.iloc[0] - g.cv_mae) / (g.cv_mae.iloc[0] - g.cv_mae.iloc[-1])
    g["gap_closed_test_%"] = 100 * (g.test_mae.iloc[0] - g.test_mae) / (g.test_mae.iloc[0] - g.test_mae.iloc[-1])
    return g

_raw = pd.DataFrame(rows)
KCURVE_ALL = pd.concat([add_gap(g) for _, g in _raw.groupby("feature_set", sort=False)], ignore_index=True)

In [ ]:
for col, lab in (("cv_mae", "CV MAE"), ("test_mae", "Test MAE"), ("gap_closed_cv_%", "Gap closed, CV (%)"), ("gap_closed_test_%", "Gap closed, test (%)")):
    print(f"\n{lab} by k and feature set")
    show(KCURVE_ALL.pivot(index="k", columns="feature_set", values=col)[list(FEATURE_SETS_COMPARE)].reset_index())
print("\nCV-selected model per feature set and k:")
show(KCURVE_ALL[["feature_set", "k", "model", "mode", "cv_mae", "test_mae", "MedAE", "R2", "MdAPE", "within_2x"]])
print("\nreference baselines (test MAE / CV MAE): " + " ; ".join(f"{b} {BASELINES[b]['MAE']:.3f} / {BASELINES[b]['cv_mae']:.3f}" for b in BASELINES))
save_table("k_curve_all_feature_sets", KCURVE_ALL, 9)

# operating point per feature set (rule applied to CV MAE only)
OP_K_BY_FS = {fs: choose_op_k({int(r.k): r.cv_mae for r in g.itertuples()}) for fs, g in KCURVE_ALL.groupby("feature_set", sort=False)}

FS_COL = dict(zip(FEATURE_SETS_COMPARE, ["#1f77b4", "#2a9d8f", "#d95f02", "#9467bd", "#8c564b", "#e377c2"]))
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, col, bkey, ttl in zip(axes, ("cv_mae", "test_mae"), ("cv_mae", "MAE"), ("CV MAE (train, expanding window)", "Test MAE (last 20% of jobs)")):
    for fs in FEATURE_SETS_COMPARE:
        g = KCURVE_ALL[KCURVE_ALL.feature_set == fs]
        ax.plot(g.k, g[col], "o-", color=FS_COL[fs], label=fs)
        ax.axvline(OP_K_BY_FS[fs], color=FS_COL[fs], ls=":", lw=1.3, alpha=.7)
    for b, ls in zip(BASELINES, ("--", "-.", ":", (0, (3, 1, 1, 1)))):
        ax.axhline(BASELINES[b][bkey], color="grey", ls=ls, lw=1.2, label=b.replace("_", " "))
    ax.set_xscale("symlog", linthresh=1); ax.set_xticks(list(K_GRID)); ax.set_xticklabels([str(k) for k in K_GRID])
    ax.set(xlabel="k = number of observed time bins (of 100)", title=ttl)
axes[0].set_ylabel("MAE on log1p(IO_TIME)"); axes[0].legend(fontsize=8, title="dotted verticals = CV operating k")
fig.suptitle("Value of information by feature set")
save_fig(fig, "k_curve_by_feature_set", 9)

In [ ]:
fs_tab = (KCURVE_ALL.groupby("feature_set", sort=False).agg(mean_cv_mae=("cv_mae", "mean"), mean_test_mae=("test_mae", "mean"))
          .reindex(list(FEATURE_SETS_COMPARE)).reset_index())
fs_tab["op_k"] = fs_tab["feature_set"].map(OP_K_BY_FS)
show(fs_tab)
# The primary feature set is the one with the lowest CV MAE averaged over K_GRID. Test columns are shown for reporting only.
FEATURE_SET = fs_tab.loc[fs_tab["mean_cv_mae"].idxmin(), "feature_set"]
print(f"\nFEATURE_SET = '{FEATURE_SET}'  (lowest mean CV MAE over K_GRID; chosen by cross-validation inside the train set only)")
save_table("feature_set_comparison", fs_tab, 10)

KCURVE = KCURVE_ALL[KCURVE_ALL.feature_set == FEATURE_SET].drop(columns="feature_set").reset_index(drop=True)
FINAL = FINAL_BY_FS[FEATURE_SET]
cv_by_k = {int(r.k): float(r.cv_mae) for r in KCURVE.itertuples()}
OP_K = choose_op_k(cv_by_k)
thr = OP_POINT_TOL * cv_by_k[max(K_GRID)]
optab = pd.DataFrame({"k": list(cv_by_k), "cv_mae": list(cv_by_k.values()), "threshold": thr, "within_tolerance": [v <= thr for v in cv_by_k.values()]})
show(optab)
OP_FINAL = FINAL[OP_K]; OP_MODELS = OP_FINAL["models"]; OP_TEST_PRED = OP_FINAL["test_pred"]
save_table("operating_point", optab, 10)

# Timeline Clustering

In [ ]:
CLUSTER_COUNTS = (5, 10, 20, 40)
SHAPE_COLS = ["read_share", "n_active", "first_active", "last_active", "peak_share", "burstiness"]

def timeline_matrix(part, k):
    cols = allowed_source_columns(k); assert_allowed_source(cols, k)
    d = part[cols]
    R = d[[f"read_{i}" for i in range(k)]].to_numpy(float); W = d[[f"write_{i}" for i in range(k)]].to_numpy(float)
    parts = []
    for ix in np.array_split(np.arange(k), min(10, k)):
        for A in (R, W):
            tot = A.sum(1); parts.append(np.divide(A[:, ix].sum(1), tot, out=np.zeros(len(part)), where=tot > 0))
    shp = build_early_features(part, k, "shape", None)[SHAPE_COLS].to_numpy(float)
    return np.column_stack(parts + [shp])

In [ ]:
def cluster_extra(Z_fit, Z_apply, c):
    sc = StandardScaler().fit(Z_fit)
    km = KMeans(n_clusters=c, n_init=10, random_state=SEED).fit(sc.transform(Z_fit))
    outs = []
    for Z in Z_apply:
        Zs = sc.transform(Z); lab = km.predict(Zs); oh = np.zeros((len(Z), c)); oh[np.arange(len(Z)), lab] = 1
        outs.append(np.column_stack([oh, km.transform(Zs).min(1)]))
    return outs

In [ ]:
def run_clustering():
    if OP_K == 0:
        print("OP_K=0: no timeline observed, clustering not applicable"); return None
    cfg = OP_FINAL; d = get_data(OP_K, FEATURE_SET)
    Z_tr, Z_te = timeline_matrix(TRAIN_DF, OP_K), timeline_matrix(TEST_DF, OP_K)
    def cv_with(c):
        errs = []
        for tr, va in FOLDS:
            X_tr, X_va = d["cv"][tr], d["cv"][va]
            if c:
                e_tr, e_va = cluster_extra(Z_tr[tr], [Z_tr[tr], Z_tr[va]], c)        # clustering fitted on the fold-train rows only
                X_tr, X_va = np.hstack([X_tr, e_tr]), np.hstack([X_va, e_va])
            est = make_estimator(cfg["model"], cfg["params"], SEED, cfg["mode"]).fit(X_tr, y_train[tr])
            errs.append(clipped_mae(y_train[va], est.predict(X_va)))
        return float(np.mean(errs))
    def test_with(c):
        X_tr, X_te = d["tr"], d["te"]
        if c:
            e_tr, e_te = cluster_extra(Z_tr, [Z_tr, Z_te], c); X_tr, X_te = np.hstack([X_tr, e_tr]), np.hstack([X_te, e_te])
            assert_features_clean(d["cols"] + [f"clu_{i}" for i in range(c)] + ["clu_dist"])
        use = SEEDS[:1] if cfg["model"] in DETERMINISTIC else SEEDS
        return clipped_mae(y_test, np.mean([make_estimator(cfg["model"], cfg["params"], s, cfg["mode"]).fit(X_tr, y_train).predict(X_te) for s in use], axis=0))
    rows = [{"variant": "no clusters", "n_clusters": 0, "cv_mae": cv_with(0), "test_mae": test_with(0)}]
    for c in CLUSTER_COUNTS:
        rows.append({"variant": f"+ K-means c={c}", "n_clusters": c, "cv_mae": cv_with(c), "test_mae": test_with(c)})
    res = pd.DataFrame(rows)
    res["cv_delta_vs_none"] = res.cv_mae - res.cv_mae.iloc[0]; res["test_delta_vs_none"] = res.test_mae - res.test_mae.iloc[0]
    show(res)
    best = res.iloc[1:].sort_values("cv_mae").iloc[0]                      # best cluster count chosen by CV only
    gain = best.cv_mae < res.cv_mae.iloc[0]
    print(f"\nbest c by CV = {int(best.n_clusters)}: CV delta {best.cv_delta_vs_none:+.4f}, test delta {best.test_delta_vs_none:+.4f} "
          f"-> {'CV improvement' if gain else 'no CV improvement'} (negative delta = lower MAE)")
    save_table("clustering_comparison", res, 12)
    return res

CLUSTER_RES = run_clustering() if RUN_CLUSTERING else None
if not RUN_CLUSTERING: print("RUN_CLUSTERING=False -> Cell 12 skipped")

# Error Analysis

In [ ]:
pred = np.clip(OP_TEST_PRED, 0, 30)
E = pd.DataFrame({"job_id": TEST_DF.job_id.to_numpy(), "INTERFACE": TEST_DF.INTERFACE.to_numpy(), "y": y_test, "pred": pred})
E["err"] = E.pred - E.y                       # signed error: prediction minus actual (log1p scale)
E["abs_err"] = E.err.abs()
E["pos"] = np.arange(len(E))

def quartile_labels(train_vals, test_vals):
    edges = np.unique(np.nanquantile(train_vals, [.25, .5, .75]))
    bins = np.concatenate([[-np.inf], edges, [np.inf]])
    labels = [f"Q{i+1} ({bins[i]:.3g}, {bins[i+1]:.3g}]" for i in range(len(bins) - 1)]
    return pd.cut(pd.Series(test_vals), bins=bins, labels=labels, right=True).astype(str).to_numpy()

E["NODES_quartile"] = quartile_labels(TRAIN_DF.NODES.to_numpy(float), TEST_DF.NODES.fillna(FIT_STATS["nodes_median"]).to_numpy(float))
if OP_K > 0:
    dd = get_data(OP_K, FEATURE_SET)
    E["bytes_sofar_quartile"] = quartile_labels(dd["tr_df"]["log_total_sofar"].to_numpy(), dd["te_df"]["log_total_sofar"].to_numpy())
else:
    E["bytes_sofar_quartile"] = "n/a (k=0)"
E["test_block"] = [f"B{i+1}: job_id {E.job_id.iloc[b].min()}-{E.job_id.iloc[b].max()}" for i, b in enumerate(np.array_split(np.arange(len(E)), 5)) for _ in b]

In [ ]:
def group_table(E, key):
    g = E.groupby(key, observed=True).agg(n_jobs=("abs_err", "size"), MAE=("abs_err", "mean"), MedAE=("abs_err", "median"),
                                          bias=("err", "mean"), abs_sum=("abs_err", "sum")).reset_index()
    g["share_of_jobs"] = g.n_jobs / len(E); g["share_of_total_error"] = g.abs_sum / E.abs_err.sum()
    return g[[key, "n_jobs", "share_of_jobs", "MAE", "MedAE", "bias", "share_of_total_error"]]

print(f"OP_K={OP_K}, model={OP_FINAL['model']}/{OP_FINAL['mode']}")
print(f"overall test: MAE={E.abs_err.mean():.3f}  MedAE={E.abs_err.median():.3f}  mean signed error (pred-actual)={E.err.mean():+.3f}")
ERR_TABLES = {}
for key, fname in (("INTERFACE", "by_interface"), ("NODES_quartile", "by_nodes_quartile"), ("bytes_sofar_quartile", "by_bytes_sofar_quartile"), ("test_block", "by_test_block")):
    t = group_table(E, key); ERR_TABLES[fname] = t
    print(f"\n--- {fname} ---"); show(t); save_table(f"error_{fname}", t, 13)

srt = E.sort_values("abs_err", ascending=False)
n5 = int(np.ceil(0.05 * len(E)))
conc = pd.DataFrame({"group": ["top-10 jobs", f"worst 5% ({n5} jobs)"], "n_jobs": [10, n5],
                     "share_of_total_abs_error": [srt.abs_err.iloc[:10].sum() / E.abs_err.sum(), srt.abs_err.iloc[:n5].sum() / E.abs_err.sum()]})
print("\n--- error concentration ---"); show(conc)
top10 = srt.head(10)[["job_id", "INTERFACE", "y", "pred", "err", "abs_err"]]
print("\n--- top-10 worst test jobs ---"); show(top10)
save_table("error_concentration", conc, 13); save_table("error_top10_jobs", top10, 13)

fig, ax = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
for itf, c in (("POSIX", "#1f77b4"), ("MPIIO", "#d95f02")):
    s = E[E.INTERFACE == itf]; ax[0].scatter(s.job_id, s.err, s=7, alpha=.5, color=c, label=itf)
ax[0].axhline(0, color="k", lw=.8); ax[0].set(ylabel="signed error (pred - actual)", title=f"Test error over job_id position (OP_K={OP_K})"); ax[0].legend()
ax[1].plot(E.job_id, E.abs_err.rolling(50, min_periods=10).mean(), color="#333", label="rolling mean |error| (50 jobs)")
ax[1].plot(E.job_id, E.abs_err.rolling(50, min_periods=10).median(), color="#2a9d8f", label="rolling median |error| (50 jobs)")
ax[1].set(xlabel="job_id (test set, time order)", ylabel="|error|"); ax[1].legend()
save_fig(fig, "error_over_position", 13)

In [ ]:
pk = pd.DataFrame({"job_id": TEST_DF.job_id.to_numpy(), "y_true": y_test})
for k in K_GRID: pk[f"pred_k{k}"] = np.clip(FINAL[k]["test_pred"], 0, 30)
save_table("test_predictions_by_k", pk, 14)
cfg = pd.DataFrame([{"SEED": SEED, "SEEDS": str(SEEDS), "N_TRIALS": N_TRIALS, "N_ITER_RS": N_ITER_RS, "K_GRID": str(K_GRID), "K_BROAD": str(K_BROAD),
                     "HEAVY_Q": HEAVY_Q, "OP_POINT_TOL": OP_POINT_TOL, "FEATURE_SET": FEATURE_SET, "OP_K": OP_K, "N_TRAIN": N_TRAIN, "N_TEST": N_TOTAL - N_TRAIN,
                     "N_FOLDS": N_FOLDS, "MIN_TRAIN_FRAC": MIN_TRAIN_FRAC, "GAP": GAP, "RUN_CLUSTERING": RUN_CLUSTERING}])
save_table("run_config", cfg, 14)
for name, t in TABLES.items():                       # re-write every registered table (idempotent)
    t.to_csv(os.path.join(OUTPUT_DIR, name + ".csv"), index=False)
mpath = os.path.join(OUTPUT_DIR, "manifest.csv"); FILES_WRITTEN[mpath] = 14
man = pd.DataFrame({"file": [os.path.basename(p) for p in FILES_WRITTEN], "written_by_cell": list(FILES_WRITTEN.values())})
man.to_csv(mpath, index=False)
missing = [p for p in FILES_WRITTEN if not os.path.exists(p)]
assert not missing, missing
print(f"{len(FILES_WRITTEN)} files in {os.path.abspath(OUTPUT_DIR)}"); show(man)

In [ ]:
def krow(k):
    r = KCURVE[KCURVE.k == k].iloc[0]
    return {"model": f"{r['model']}/{r['mode']}", "cv_mae": round(r["cv_mae"], 3), "test_mae": round(r["test_mae"], 3), "test_MedAE": round(r["MedAE"], 3),
            "test_R2": round(r["R2"], 3), "test_MdAPE": round(r["MdAPE"], 3), "test_within_2x": round(r["within_2x"], 3)}

hz = HEAVY[HEAVY.k == OP_K].iloc[0]
SUMMARY = {
    "dataset": {"n_jobs": N_TOTAL, "n_train": N_TRAIN, "n_test": N_TOTAL - N_TRAIN, "last_train_job_id": int(TRAIN_DF.job_id.iloc[-1]), "first_test_job_id": int(TEST_DF.job_id.iloc[0])},
    "run": {"FEATURE_SET": FEATURE_SET, "OP_K": OP_K, "K_GRID": list(K_GRID), "SEEDS": list(SEEDS), "N_TRIALS": N_TRIALS},
    "feature_set_comparison": {"selected_by_cv": FEATURE_SET, "mean_cv_mae_over_K_GRID": {r.feature_set: round(r.mean_cv_mae, 3) for r in fs_tab.itertuples()},
                               "mean_test_mae_over_K_GRID": {r.feature_set: round(r.mean_test_mae, 3) for r in fs_tab.itertuples()}, "op_k_by_feature_set": OP_K_BY_FS},
    "baselines_test_mae": {"global_median": round(BASELINES["global_median"]["MAE"], 3), "global_mean": round(BASELINES["global_mean"]["MAE"], 3)},
    "k=0": krow(0), f"k=OP_K({OP_K})": krow(OP_K), f"k={max(K_GRID)}": krow(max(K_GRID)),
    "best_model_per_k": {int(k): f"{m}/{md}" for k, m, md in zip(KCURVE.k, KCURVE.model, KCURVE["mode"])},
    "heavy_io_at_OP_K": {"classifier": hz["classifier"], "heavy_q": HEAVY_Q, "cv_pr_auc": round(hz["cv_pr_auc"], 3), "roc_auc": round(hz["roc_auc"], 3), "pr_auc": round(hz["pr_auc"], 3),
                         "precision_at_10pct": round(hz["precision_at_10pct"], 3), "recall_at_10pct": round(hz["recall_at_10pct"], 3),
                         "train_prevalence": round(hz["train_prevalence"], 3), "test_prevalence": round(hz["test_prevalence"], 3)},
    "files_written": [os.path.basename(p) for p in FILES_WRITTEN],
}